# Cricket Bat & Ball Detection Model Training (YOLOv8)
### Google Colab / Kaggle Notebook

This notebook trains a high-speed YOLOv8 detector for:
- **Cricket Bat** (tracking trajectory, face angle, impact point)
- **Cricket Ball** (tracking trajectory, speed, contact frame)
- **Batsman & Bowler** (spatial localization)

Data source: Roboflow Universe (`cricket-2-7snzq`, `cricket-bat-detection-hlbdq`, `cricket-ball-detection-4g7e0`).
Credentials are read securely from `ROBOFLOW_API_KEY`.

In [ ]:
# 1. Environment Setup & Dependency Installation
!nvidia-smi
!pip install -q ultralytics roboflow onnx onnxruntime tensorflowjs

In [ ]:
# 2. Configure Environment Variables & Roboflow Credentials
import os
from roboflow import Roboflow

# Set your Roboflow API key via environment variable or prompt
ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY")
if not ROBOFLOW_API_KEY:
    from getpass import getpass
    ROBOFLOW_API_KEY = getpass("Enter your Roboflow API key: ")
    os.environ["ROBOFLOW_API_KEY"] = ROBOFLOW_API_KEY

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
# Download Cricket-2 dataset containing bat, ball, batsman, bowler in YOLOv8 format
project = rf.workspace("lbwdrs").project("cricket-2-7snzq")
dataset = project.version(1).download("yolov8")
print("Dataset downloaded to:", dataset.location)

In [ ]:
# 3. Verify data.yaml path
import yaml
from pathlib import Path

data_yaml_path = Path(dataset.location) / "data.yaml"
with open(data_yaml_path, "r") as f:
    data_config = yaml.safe_load(f)

print("Classes in dataset:", data_config.get("names"))
print("Number of classes:", data_config.get("nc"))

In [ ]:
# 4. Initialize YOLOv8 Model (Pretrained Nano/Small for high FPS inference)
from ultralytics import YOLO

# Load lightweight YOLOv8n or YOLOv8s for 60+ FPS inference
model = YOLO("yolov8n.pt")

# Train on GPU
results = model.train(
    data=str(data_yaml_path),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    workers=4,
    name="cricket_bat_ball_yolov8",
    # Hyperparameters tuned for fast-moving sports objects
    degrees=10.0,
    translate=0.1,
    scale=0.2,
    mosaic=1.0,
    mixup=0.1,
    patience=15
)

In [ ]:
# 5. Validation and Evaluation Metrics
val_metrics = model.val()
print("Validation mAP50:", val_metrics.box.map50)
print("Validation mAP50-95:", val_metrics.box.map)

In [ ]:
# 6. Export to ONNX (for Python/FastAPI backend) & TF.js (for browser direct inference)
onnx_file = model.export(format="onnx", dynamic=True, simplify=True)
print("Exported ONNX model:", onnx_file)

# Export to web format if needed for client edge execution
try:
    tflite_file = model.export(format="tflite")
    print("Exported TFLite model:", tflite_file)
except Exception as e:
    print("TFLite export notice:", e)